In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_Library_Small_DLD1_PE2max():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1


def load_branch2_data_Library_Small_DLD1_PE2max():
    """
    Loads (90,3) data from "Feature_CNN2_reduced.txt".
    """
    data_branch2 = []
    current_array = []
    with open("Feature_CNN2_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch2.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch2.append(current_array)
    X_branch2 = np.array(data_branch2)
    X_branch2 = X_branch2.reshape(len(X_branch2), 90, 3)
    return X_branch2


def load_branch3_data_Library_Small_DLD1_PE2max(filename="Feature_MLP_expanded.txt"):
    """
    Loads 120-dimensional numeric features from file.
    Each line has 120 floats (space- or comma-delimited).
    """
    X_branch3 = []
    with open(filename, 'r') as file:
        for line in file:
            line = line.strip()
            # Expect 120 numbers per line
            values = line.replace('[', '').replace(']', '').split()
            float_vals = [float(v.strip().replace(',', '')) for v in values]
            X_branch3.append(float_vals)
    
    X_branch3 = np.array(X_branch3)  # shape: (n_samples, 120)
    X_branch3 = X_branch3.reshape(len(X_branch3), 120)
    return X_branch3


def load_reaction_rates_Library_Small_DLD1_PE2max():
    with open('Library_Small_DLD1_PE2max_filtered_NGG_editing_efficiency_value_percentage.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


########################################
# 2. Graph Data Utilities (for GNN)
########################################

def matrix_to_edge_index(prob_matrix):
    edges_source = []
    edges_target = []
    L = len(prob_matrix)
    for i in range(L):
        for j in range(i + 1, L):  # only i < j
            if prob_matrix[i][j] > 0.01:
                edges_source.append(i)
                edges_target.append(j)
    return [edges_source, edges_target]

def matrix_to_edge_index_collapsed(prob_matrix: np.ndarray, threshold: float = 0.01):
    """
    Build undirected edges (i < j) from a dense probability matrix using a threshold.
    Returns edge_index as [sources, targets] lists of ints.
    """
    src, dst = [], []
    L = prob_matrix.shape[0]
    for i in range(L):
        for j in range(i+1, L):
            if prob_matrix[i, j] > threshold:
                src.append(i)
                dst.append(j)
    return [src, dst]

def generate_edge_features(edge_index, prob_matrix):
    if not edge_index:
        return []
    feats = []
    for i in range(len(edge_index[0])):
        src = edge_index[0][i]; tgt = edge_index[1][i]
        feats.append([prob_matrix[src][tgt]])  # shape [E, 1]
    return feats

def _num_nodes_from_edge_index(edge_index_list, fallback):
    if not edge_index_list or len(edge_index_list[0]) == 0:
        return fallback
    mx = 0
    for a in edge_index_list:
        if len(a) > 0:
            mx = max(mx, max(a))
    return mx + 1


def collapse_to_first_k_plus_1_sum_Library_Small_DLD1_PE2max(prob_matrix: np.ndarray, first_k: int = 20) -> np.ndarray:
    """
    Collapse an LxL base-pair probability matrix into (first_k + 1) x (first_k + 1):
      - Rows/cols 0..first_k-1: original 1..first_k bases
      - Row/col first_k: an 'outside' super-node representing bases >= first_k
    For each i in [0..first_k-1], the edge prob to the super-node is the
    SUM of probabilities of pairing to bases >= first_k.
    """
    L = prob_matrix.shape[0]
    K = first_k
    new_size = K + 1
    newP = np.zeros((new_size, new_size), dtype=float)

    # Copy the 0..K-1 block
    newP[:K, :K] = prob_matrix[:K, :K][::-1, ::-1]

    if L > K:
        # Outside block (K..L-1)
        outside_block = prob_matrix[:K, K:L]  # shape: K x (L-K)
        # Sum of probabilities across outside bases
        p_sum = np.sum(outside_block, axis=1)
        # Fill symmetric connections to the super-node K
        newP[:K, K] = p_sum
        newP[K, :K] = p_sum

    # Leave newP[K, K] as 0
    return newP

def slice_and_renumber_duplex_Library_Small_DLD1_PE2max(prob_matrix, len_guide, len_target, k=20):
    """
    Keep only the first k positions of the guide and the last k positions of the target,
    then reverse the order within each block (guide: k→1, target: 2k→k+1).
    Returns a (kg+kt) x (kg+kt) reduced/renumbered matrix.
    """
    L = len_guide + len_target
    assert prob_matrix.shape == (L, L), "prob_matrix size mismatch."

    kg = min(k, len_guide)
    kt = min(k, len_target)

    # Original indices: guide = [0..len_guide-1], target = [len_guide..L-1]
    sel_g = np.arange(0, kg)                      # first k of guide
    sel_t = np.arange(L - kt, L)                  # last k of target
    S = np.concatenate([sel_g, sel_t])

    # Extract submatrix for these rows/cols
    P = prob_matrix[np.ix_(S, S)].copy()

    # Reverse guide rows/cols
    P[:kg, :] = P[:kg, :][::-1, :]
    P[:, :kg] = P[:, :kg][:, ::-1]

    return P


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


def load_dna_reaction_data_Library_Small_DLD1_PE2max():
    with open('Library_Small_DLD1_PE2max_filtered_NGG_full_guide_sequences.txt', 'r') as f:
        crna_sequences = [line.strip() for line in f.readlines()]
    with open('Library_Small_DLD1_PE2max_filtered_NGG_target_sequences_noPAM.txt', 'r') as f:
        target_sequences = [line.strip() for line in f.readlines()]

    FIRST_K = 20  # keep first 20 bases, collapse the rest into node index 20 (21st base)
        
    crna_edge_indices = []
    crna_edge_features = []
    for guide in crna_sequences:
        prob_full = pairs(strands=guide, model=my_model_RNA).to_array()
        prob_collapsed = collapse_to_first_k_plus_1_sum_Library_Small_DLD1_PE2max(prob_full, first_k=FIRST_K)
        edge_index = matrix_to_edge_index_collapsed(prob_collapsed, threshold=0.01)
        feats = generate_edge_features(edge_index, prob_collapsed)
        crna_edge_indices.append(edge_index)   # each is [sources, targets] with 0..20 node indices
        crna_edge_features.append(feats)       # [[p_ij], ...]

    ssDNA_target_bh_edge_indices = []
    ssDNA_target_bh_edge_features = []
    for i in range (0, len(crna_sequences)):
        prob_matrix = pairs(strands=target_sequences[i], model=my_model_DNA).to_array()
        edge_index = matrix_to_edge_index(prob_matrix)
        feats = generate_edge_features(edge_index, prob_matrix)
        ssDNA_target_bh_edge_indices.append(edge_index)
        ssDNA_target_bh_edge_features.append(feats)

    
    duplex_edge_indices = []
    duplex_edge_features = []
    
    for i in range (0, len(crna_sequences)):
        prob_matrix = pairs(strands=[crna_sequences[i], target_sequences[i]], model=my_model_RNA).to_array()
        len_g, len_t = len(crna_sequences[i]), len(target_sequences[i])
        # Crop to guide[1..20] + target[last 20], renumber both reversed
        updated_P = slice_and_renumber_duplex_Library_Small_DLD1_PE2max(prob_matrix, len_g, len_t, k=20)
        edge_index = matrix_to_edge_index(updated_P)
        feats = generate_edge_features(edge_index, updated_P)
        duplex_edge_indices.append(edge_index)
        duplex_edge_features.append(feats)

    
    with open('Library_Small_DLD1_PE2max_filtered_NGG_editing_efficiency_value_percentage.txt', 'r') as f:
        reaction_rates = [float(line.strip()) for line in f.readlines()]
    
    return (crna_edge_indices, ssDNA_target_bh_edge_indices, duplex_edge_indices, 
            crna_edge_features, ssDNA_target_bh_edge_features, duplex_edge_features, 
            reaction_rates)


from typing import Iterable, List, Tuple, Any

# Each graph set is a tuple:
# (crna_ei, tbh_ei, d_ei, crna_ea, tbh_ea, d_ea, y)
GraphSet = Tuple[Iterable[Any], Iterable[Any], Iterable[Any],
                 Iterable[Any], Iterable[Any], Iterable[Any], Iterable[Any]]

def combine_dna_graph_sets(*graph_sets: GraphSet):
    """Concatenate any number of DNA/RNA graph sets along the sample axis.
       Optionally return a source label per sample (0,1,2,...) indicating origin."""
    if not graph_sets:
        raise ValueError("Provide at least one graph set")

    c_ei: List[Any] = []
    tbh_ei: List[Any] = []
    d_ei: List[Any] = []
    c_ea: List[Any] = []
    tbh_ea: List[Any] = []
    d_ea: List[Any] = []
    y: List[Any] = []

    for sid, gs in enumerate(graph_sets):
        try:
            c_ei_a, tbh_ei_a, d_ei_a, c_ea_a, tbh_ea_a, d_ea_a, y_a = gs
        except Exception as e:
            raise ValueError(f"Graph set #{sid} must be a 7-tuple") from e

        # Optional consistency check per set
        n = len(y_a)
        if not all(len(lst) == n for lst in (c_ei_a, tbh_ei_a, d_ei_a, c_ea_a, tbh_ea_a, d_ea_a)):
            raise ValueError(f"Graph set #{sid} has mismatched lengths")

        c_ei.extend(list(c_ei_a))
        tbh_ei.extend(list(tbh_ei_a))
        d_ei.extend(list(d_ei_a))
        c_ea.extend(list(c_ea_a))
        tbh_ea.extend(list(tbh_ea_a))
        d_ea.extend(list(d_ea_a))
        y.extend(list(y_a))

    combined = (c_ei, tbh_ei, d_ei, c_ea, tbh_ea, d_ea, y)
    return combined


class CRNADuplexSubstructuresDataset(Dataset):
    def __init__(self, crna_edge_indices, ssDNA_target_bh_edge_indices, duplex_edge_indices, 
                 crna_edge_features, ssDNA_target_bh_edge_features, duplex_edge_features, 
                 reaction_rates):
        super().__init__()
        self.crna_edge_indices = crna_edge_indices
        self.ssDNA_target_bh_edge_indices = ssDNA_target_bh_edge_indices
        self.duplex_edge_indices = duplex_edge_indices
        self.crna_edge_features = crna_edge_features
        self.ssDNA_target_bh_edge_features = ssDNA_target_bh_edge_features
        self.duplex_edge_features = duplex_edge_features
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)
    def __len__(self):
        return self.num_samples
    def __getitem__(self, idx):
        c_edge_index_list = self.crna_edge_indices[idx]
        if not c_edge_index_list:
            c_edge_index = torch.empty((2,0), dtype=torch.long)
            n_c = 20  # fallback (e.g., 20-nt)
        else:
            c_edge_index = torch.tensor(c_edge_index_list, dtype=torch.long)
            n_c = _num_nodes_from_edge_index(c_edge_index_list, fallback=20)
        c_x = torch.zeros((n_c, 4), dtype=torch.float)  # placeholder
        c_edge_attr = torch.tensor(self.crna_edge_features[idx], dtype=torch.float)
        if c_edge_attr.dim() == 1:
            c_edge_attr = c_edge_attr.unsqueeze(1)   # [E,1]
        cRNA_data = Data(x=c_x, edge_index=c_edge_index, edge_attr=c_edge_attr)

        t_bh_edge_index_list = self.ssDNA_target_bh_edge_indices[idx]
        if not t_bh_edge_index_list:
            t_bh_edge_index = torch.empty((2,0), dtype=torch.long)
            n_t = 20
        else:
            t_bh_edge_index = torch.tensor(t_bh_edge_index_list, dtype=torch.long)
            n_t = _num_nodes_from_edge_index(t_bh_edge_index_list, fallback=20)
        t_bh_x = torch.zeros((n_t, 4), dtype=torch.float)
        t_bh_edge_attr = torch.tensor(self.ssDNA_target_bh_edge_features[idx], dtype=torch.float)
        if t_bh_edge_attr.dim() == 1:
            t_bh_edge_attr = t_bh_edge_attr.unsqueeze(1)   # [E,1]
        ssDNA_target_bh_data = Data(x=t_bh_x, edge_index=t_bh_edge_index, edge_attr=t_bh_edge_attr)
        
        d_edge_index_list = self.duplex_edge_indices[idx]
        if not d_edge_index_list:
            d_edge_index = torch.empty((2,0), dtype=torch.long)
            n_d = 40
        else:
            d_edge_index = torch.tensor(d_edge_index_list, dtype=torch.long)
            n_d = _num_nodes_from_edge_index(d_edge_index_list, fallback=40)
        d_x = torch.zeros((n_d, 4), dtype=torch.float)
        d_edge_attr = torch.tensor(self.duplex_edge_features[idx], dtype=torch.float)
        if d_edge_attr.dim() == 1:
            d_edge_attr = d_edge_attr.unsqueeze(1)   # [E,1]
        duplex_data = Data(x=d_x, edge_index=d_edge_index, edge_attr=d_edge_attr)
        
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return cRNA_data, ssDNA_target_bh_data, duplex_data, y_val

def graph_collate(batch):
    from torch_geometric.data import Batch
    cRNA_list, ssDNA_target_bh_list, duplex_list, y_list = zip(*batch)
    batch_cRNA = Batch.from_data_list(cRNA_list)
    batch_ssDNA_target_bh = Batch.from_data_list(ssDNA_target_bh_list)
    batch_duplex = Batch.from_data_list(duplex_list)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return batch_cRNA, batch_ssDNA_target_bh, batch_duplex, y



########################################
# 3. CNN, GNN, MLP Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x
        

class CNNBranch2(nn.Module):
    """
    CNN branch for (90,3).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(3, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*45, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,3,45)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,45)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import NNConv, global_mean_pool, global_max_pool


class GNNBranch(nn.Module):
    def __init__(self, hidden_dim=32, edge_attr_dim=1):
        super().__init__()
        self.hidden_dim = hidden_dim

        def make_edge_mlp(edge_attr_dim, in_channels, out_channels):
            return nn.Sequential(
                nn.Linear(edge_attr_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, in_channels * out_channels)
            )

        self.edge_mlp_c   = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_c2  = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_c = NNConv(4, hidden_dim, self.edge_mlp_c, aggr='mean')
        self.conv2_c = NNConv(hidden_dim, hidden_dim, self.edge_mlp_c2, aggr='mean')

        self.edge_mlp_bh  = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_bh2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_t_bh = NNConv(4, hidden_dim, self.edge_mlp_bh, aggr='mean')
        self.conv2_t_bh = NNConv(hidden_dim, hidden_dim, self.edge_mlp_bh2, aggr='mean')

        self.edge_mlp_d   = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_d2  = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_d = NNConv(4, hidden_dim, self.edge_mlp_d, aggr='mean')
        self.conv2_d = NNConv(hidden_dim, hidden_dim, self.edge_mlp_d2, aggr='mean')

        self.mlp_merge = nn.Sequential(
            nn.Linear(hidden_dim * 3 * 2, hidden_dim),  # 3 graphs, mean+max
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def pool(self, x, batch):
        return torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)

    def forward(self, crna_data, ssDNA_target_bh_data, duplex_data):
        def process(graph_data, conv1, conv2):
            x, edge_index, edge_attr, batch = graph_data.x, graph_data.edge_index, graph_data.edge_attr, graph_data.batch
            x = F.elu(conv1(x, edge_index, edge_attr))   # << correct order
            x = F.elu(conv2(x, edge_index, edge_attr))
            return self.pool(x, batch)

        x_c   = process(crna_data, self.conv1_c,   self.conv2_c)
        x_tbh = process(ssDNA_target_bh_data, self.conv1_t_bh, self.conv2_t_bh)
        x_d   = process(duplex_data, self.conv1_d, self.conv2_d)

        merged = torch.cat([x_c, x_tbh, x_d], dim=1)
        return self.mlp_merge(merged)


class MLPBranch120(nn.Module):
    """
    A single hidden layer MLP for 120D => output dimension mlp_dim.
    No separate "output" layer; just one fc + ReLU => final embedding.
    """
    def __init__(self, input_dim=120, hidden_dim=32):
        super().__init__()
        self.fc = nn.Linear(input_dim, hidden_dim)
    def forward(self, x):
        # x shape: (batch,120)
        x = F.relu(self.fc(x))  # (batch,hidden_dim)
        return x

########################################
# 4. Final Fusion Model
########################################

class CNN_GNN_MLP_Fusion(nn.Module):
    """
    End-to-end: 
      - CNNBranch1 => feat1
      - CNNBranch2 => feat2
      - GNNBranch  => feat_gnn
      - MLPBranch120 => feat_mlp
    Concat => dropout => final FC => 1
    """
    def __init__(self,
                 filters1, kernel_size1, dense_units1,  # CNN1
                 filters2, kernel_size2, dense_units2,  # CNN2
                 gnn_hidden_dim,
                 mlp_hidden_dim,  # single hidden dimension for MLP
                 final_fc_dim,
                 dropout_rate=0.0):  # new hyperparameter for dropout
        super().__init__()
        
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.cnn_branch2 = CNNBranch2(filters2, kernel_size2, dense_units2)
        self.gnn_branch = GNNBranch(hidden_dim=gnn_hidden_dim)
        self.mlp_branch = MLPBranch120(input_dim=120, hidden_dim=mlp_hidden_dim)
        
        # total dimension = (dense_units1 + dense_units2 + gnn_hidden_dim + mlp_hidden_dim)
        total_dim = dense_units1 + dense_units2 + gnn_hidden_dim + mlp_hidden_dim
        
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(total_dim, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    
    def forward(self, crna_data, ssDNA_target_bh_data, duplex_data, x1, x2, x3):
        feat1 = self.cnn_branch1(x1)                   # (batch, dense_units1)
        feat2 = self.cnn_branch2(x2)                   # (batch, dense_units2)
        feat_gnn = self.gnn_branch(crna_data, ssDNA_target_bh_data, duplex_data)  # (batch, gnn_hidden_dim)
        feat_mlp = self.mlp_branch(x3)                 # (batch, mlp_hidden_dim)
        
        merged = torch.cat([feat1, feat2, feat_gnn, feat_mlp], dim=1)
        # Apply dropout on the concatenated features
        merged = F.dropout(merged, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))   # (batch, final_fc_dim)
        out = self.out(x)                 # (batch, 1)
        return out.view(-1)



########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, graph_dataset, X1, X2, X3, reaction_rates):
        super().__init__()
        self.graph_dataset = graph_dataset
        self.X1 = X1
        self.X2 = X2
        self.X3 = X3
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)
        
        assert len(graph_dataset) == self.num_samples
        assert len(X1) == self.num_samples
        assert len(X2) == self.num_samples
        assert len(X3) == self.num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        crna_data, ssDNA_target_bh_data, duplex_data, y_val = self.graph_dataset[idx]
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)  
        x2 = torch.tensor(self.X2[idx], dtype=torch.float)  
        x3 = torch.tensor(self.X3[idx], dtype=torch.float)  
        return crna_data, ssDNA_target_bh_data, duplex_data, x1, x2, x3, y_val

def hybrid_collate(batch):
    from torch_geometric.data import Batch
    cRNA_list, ssDNA_target_bh_list, duplex_list, x1_list, x2_list, x3_list, y_list = zip(*batch)
    batch_cRNA = Batch.from_data_list(cRNA_list)
    batch_ssDNA_target_bh = Batch.from_data_list(ssDNA_target_bh_list)
    batch_duplex = Batch.from_data_list(duplex_list)
    x1 = torch.stack(x1_list, dim=0)
    x2 = torch.stack(x2_list, dim=0)
    x3 = torch.stack(x3_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return batch_cRNA, batch_ssDNA_target_bh, batch_duplex, x1, x2, x3, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # Rebuild for final 100-trial evaluation by loading the entire saved model
    model = torch.load(f"Generalist_model.pt", weights_only=False)
    model.eval()
    
    # final hold-out evaluation
    # load data

    X1_Library_Small_DLD1_PE2max = load_branch1_data_Library_Small_DLD1_PE2max()
    X1_Library_Small_DLD1_PE2max   = np.asarray(X1_Library_Small_DLD1_PE2max)
    X1 = np.concatenate([X1_Library_Small_DLD1_PE2max], axis=0) 

    
    X2_Library_Small_DLD1_PE2max = load_branch2_data_Library_Small_DLD1_PE2max()
    X2_Library_Small_DLD1_PE2max   = np.asarray(X2_Library_Small_DLD1_PE2max)
    X2 = np.concatenate([X2_Library_Small_DLD1_PE2max], axis=0) 

    
    X3_Library_Small_DLD1_PE2max = load_branch3_data_Library_Small_DLD1_PE2max()
    X3_Library_Small_DLD1_PE2max   = np.asarray(X3_Library_Small_DLD1_PE2max)
    X3 = np.concatenate([X3_Library_Small_DLD1_PE2max], axis=0) 


    rates_Library_Small_DLD1_PE2max = load_reaction_rates_Library_Small_DLD1_PE2max()
    rates_Library_Small_DLD1_PE2max   = np.asarray(rates_Library_Small_DLD1_PE2max)
    rates = np.concatenate([rates_Library_Small_DLD1_PE2max], axis=0) 
    
    # Load both sources
    data_Library_Small_DLD1_PE2max   = load_dna_reaction_data_Library_Small_DLD1_PE2max()
    
    # Combine
    combined = combine_dna_graph_sets(data_Library_Small_DLD1_PE2max)
    
    # Unpack if needed
    (crna_edge_indices, ssDNA_target_bh_edge_indices, duplex_edge_indices,
     crna_edge_features, ssDNA_target_bh_edge_features, duplex_edge_features,
     reaction_rates) = combined
    
    graph_dataset = CRNADuplexSubstructuresDataset(
        crna_edge_indices, ssDNA_target_bh_edge_indices, duplex_edge_indices, 
        crna_edge_features, ssDNA_target_bh_edge_features, duplex_edge_features, 
        rates
    )
    hybrid_dataset = HybridDataset(graph_dataset, X1, X2, X3, rates)

    # Get unseen Library_Small_DLD1_PE2max dataset
    np.random.seed(42)
    full_indices_Library_Small_DLD1_PE2max = np.arange(len(rates_Library_Small_DLD1_PE2max))
    selected_indices_Library_Small_DLD1_PE2max = np.random.choice(len(full_indices_Library_Small_DLD1_PE2max), size=len(full_indices_Library_Small_DLD1_PE2max), replace=False)
    unseen_indices_Library_Small_DLD1_PE2max = np.setdiff1d(full_indices_Library_Small_DLD1_PE2max, selected_indices_Library_Small_DLD1_PE2max)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_Library_Small_DLD1_PE2max = Subset(hybrid_dataset, unseen_indices_Library_Small_DLD1_PE2max)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_Library_Small_DLD1_PE2max = Subset(hybrid_dataset, selected_indices_Library_Small_DLD1_PE2max)
    trial_loader = DataLoader(selected_set_Library_Small_DLD1_PE2max, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    preds_trial = []
    labels_trial = []
    with torch.no_grad():
        for c_b, tbh_b, d_b, xx1_b, xx2_b, xx3_b, yy_b in trial_loader:
            p = model(c_b, tbh_b, d_b, xx1_b, xx2_b, xx3_b)
            preds_trial.append(p.item())
            labels_trial.append(yy_b.item())
        sp_corr, _ = spearmanr(labels_trial, preds_trial)
        print(sp_corr)
        print(f"Average Spearman Correlation for the selected dataset: {sp_corr}")

    with open("preds_Generalist.txt", 'w') as f:
        for val in preds_trial:
            f.write(f"{val}\n")
    

if __name__ == "__main__":
    main_pipeline() 